In [ ]:
# Install necessary dependencies
import json
import os
import sys

from dotenv import load_dotenv

# Ensure the root directory is in the path so 'src' is recognizable
sys.path.append(os.path.abspath(".."))

from src.mock_tools import SerpManager
from src.loader import TaskLoader
from src.baseline import DirectBaseline, SerpAgent
from src.evaluator import NomadEvaluator
from src.utils import *

# Create data directory if it doesn't exist
os.makedirs("data", exist_ok=True)
load_dotenv()

In [ ]:
# --- Setup ---
load_dotenv()
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

loader = TaskLoader("data/tasks.json")
evaluator = NomadEvaluator("data/tasks.json")

# --- Test T1-01 ---
task_id = "T1-01"
task = loader.get_task(task_id)
context = loader.format_for_agent(task)

print(f"🚀 Running Direct-Prompting Baseline: {task_id}")
agent = DirectBaseline(api_key=ANTHROPIC_KEY)

# No tool calls, no Cache Misses!
itinerary, logs = agent.solve(context)
# --- Print Agent's Raw Result ---
print("\n" + "=" * 20 + " AGENT OUTPUT " + "=" * 20)
print(itinerary)
print("=" * 54)

# --- Evaluate ---
results = evaluator.evaluate(task_id, {"itinerary": itinerary}, logs)

print("\n" + "=" * 40)
print(f"📊 DIRECT BASELINE RESULTS")
print(f"CSR (Constraint Satisfaction): {results['csr']:.2%}")
print(f"Tool Accuracy: {results['tool_accuracy']:.2%}")  # Will be 0% by design
print(f"Constraints Audit: {results['details']['constraint_breakdown']}")
print("=" * 40)

save_experiment_result(task["task_id"], "DirectBaseline", itinerary, logs, results)

In [ ]:
# 1. Environment Setup
load_dotenv()
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")
SERP_KEY = os.getenv("SERPAPI_KEY")

# 2. Initialize Components
loader = TaskLoader("data/tasks.json")
# manager uses 'mock' to ensure we don't spend money during testing
manager = SerpManager(api_key=SERP_KEY, cache_path="data/serp_snapshot.json")
agent = SerpAgent(api_key=ANTHROPIC_KEY, tool_provider=manager)
evaluator = NomadEvaluator("data/tasks.json")
# 3. Execute Task
task_id = "T1-01"
print(f"🚀 Running {task_id} in Auto Mode...")
task = loader.get_task(task_id)
context = loader.format_for_agent(task)

try:
    itinerary, logs = agent.solve(context)

    print("\n" + "=" * 30 + " AGENT RECOMMENDATION " + "=" * 30)
    print(itinerary)
    print("=" * 82)
    print(f"Tools Used: {logs}")

    # --- Evaluate ---
    results = evaluator.evaluate(task_id, {"itinerary": itinerary}, logs)

    print("\n" + "=" * 40)
    print(f"📊 DIRECT BASELINE RESULTS")
    print(f"CSR (Constraint Satisfaction): {results['csr']:.2%}")
    print(f"Tool Accuracy: {results['tool_accuracy']:.2%}")  # Will be 0% by design
    print(f"Constraints Audit: {results['details']['constraint_breakdown']}")
    print("=" * 40)

except Exception as e:
    print(f"❌ Error: {e}")